In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from collections import Counter

In [2]:
data = pd.read_csv('../../data/train.csv.zip', compression='zip')
data.shape

(109237, 2)

In [3]:
attributes = data.attribute_ids.tolist()
attributes = ' '.join(attributes)
attributes = attributes.split()
attributes = Counter(attributes)

In [7]:
subset = pd.read_csv('../../data/subset.csv')
tail_1 = subset[(subset['percent'] <= 0.2) & (subset['total'] <= 200)]
tail_2 = subset[subset['total'] <= 20]
long_tail = tail_1.append(tail_2)['intent'].unique().tolist()
long_tail = [str(x) for x in long_tail]
len(long_tail)

570

In [5]:
def clean_labels(classes):
    classes = classes.split()
    classes = [x for x in classes if x not in long_tail]
    classes = ' '.join(classes)
    return classes

def rare_class(classes):
    classes = classes.split()
    values = [attributes[x] for x in classes]
    value = np.argmin(values)
    return classes[value]

In [6]:
data['attribute_ids'] = data['attribute_ids'].map(lambda x : clean_labels(x))
data = data[data['attribute_ids'].str.len() > 0]
data = data.reset_index(drop=True)
data['class'] = data['attribute_ids'].map(lambda x : rare_class(x))

In [7]:
folds = StratifiedKFold(n_splits=10, shuffle=False, random_state=2017)

In [8]:
counter = 0
data['fold'] = 0
for _, idx in folds.split(data.index, data['class']):
    counter += 1
    data.loc[idx ,'fold'] = counter

/opt/conda/lib/python3.7/site-packages/sklearn/model_selection/_split.py:657: Warning: The least populated class in y has only 1 members, which is too few. The minimum number of members in any class cannot be less than n_splits=10.
  % (min_groups, self.n_splits)), Warning)


In [9]:
data.to_csv('../../data/folds.csv', index=False)

In [10]:
data['fold'].value_counts()

1     11124
2     11068
3     11012
4     10960
5     10904
6     10856
7     10794
8     10757
9     10703
10    10651
Name: fold, dtype: int64

In [11]:
attributes = data.attribute_ids.tolist()
attributes = ' '.join(attributes)
attributes = attributes.split()
attributes = Counter(attributes)
print(len(attributes))

533
